# SQL Alert Manager (Alerts V2)
Creates Databricks SQL **Alerts V2** programmatically via `POST /api/2.0/alerts`.
Reads alert queries from `alerts.sql`, creates V2 alerts with inline queries,
evaluation conditions (COUNT > 0), email subscribers, and a cron schedule.
Also includes a demo alert that always triggers for live presentations.

In [0]:
import os, re, requests

# ── Alert configuration ───────────────────────────────────────
ALERT_CONFIG = {
    "warehouse_id": "960eea0736a2e75b",
    "notify_email": "siva.borusu@jrvs.ca",
    # 30 min after the Quality Monitor refresh (08:00 AM ET)
    "schedule_cron": "0 30 8 * * ?",
    "schedule_timezone": "America/Toronto",
    "alerts_sql_path": os.path.join(os.getcwd(), "alerts.sql"),
    "profile_table": "mlops.siva_borusu.customer_n_transactions_drift_profile_metrics",
}

In [0]:
class AlertManager:
    """Creates Databricks SQL Alerts V2 from alert definitions in alerts.sql.

    Uses the Alerts V2 REST API:
      POST   /api/2.0/alerts          — create alert (inline query, evaluation, subscriber)
      GET    /api/2.0/alerts          — list alerts
      DELETE /api/2.0/alerts/{id}     — trash alert

    V2 alerts embed the query text directly (no separate saved query).
    """

    API = "/api/2.0/alerts"

    def __init__(self, config: dict):
        self.cfg = config
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        token = ctx.apiToken().get()
        self.host = f"https://{ctx.browserHostName().get()}"
        self._headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }

    # ── REST helper ──────────────────────────────────────────
    def _api(self, method: str, path: str, body: dict = None) -> dict:
        resp = requests.request(
            method, f"{self.host}{path}", headers=self._headers, json=body,
        )
        if not resp.ok:
            raise Exception(f"API {resp.status_code}: {resp.text[:500]}")
        return resp.json() if resp.content else {}

    # ── Parse alerts.sql ─────────────────────────────────────
    def _parse_alerts_sql(self) -> list[dict]:
        """Split alerts.sql on '-- Alert N:' markers into named queries."""
        with open(self.cfg["alerts_sql_path"]) as f:
            content = f.read()
        blocks = re.split(r"\n\n+(?=-- Alert \d+:)", content)
        alerts = []
        for block in blocks:
            block = block.strip()
            if not block.startswith("-- Alert"):
                continue
            lines = block.split("\n")
            name = re.sub(r"^-- Alert \d+:\s*", "", lines[0]).strip()
            name = name.replace("/", "-")  # workspace names cannot contain '/'
            sql_lines = [l for l in lines if not l.strip().startswith("--")]
            query_text = "\n".join(sql_lines).strip().rstrip(";")
            if query_text:
                alerts.append({"name": name, "query_text": query_text})
        return alerts

    # ── Build V2 alert payload ────────────────────────────────
    def _alert_body(self, name: str, query_text: str,
                    source_col: str = "window_start") -> dict:
        return {
            "display_name": name,
            "query_text": query_text,
            "warehouse_id": self.cfg["warehouse_id"],
            "evaluation": {
                "source": {
                    "aggregation": "COUNT",
                    "name": source_col,
                },
                "comparison_operator": "GREATER_THAN",
                "threshold": {"value": {"double_value": 0}},
                "notification": {
                    "subscriptions": [
                        {"user_email": self.cfg["notify_email"]}
                    ],
                },
            },
            "schedule": {
                "quartz_cron_schedule": self.cfg["schedule_cron"],
                "timezone_id": self.cfg["schedule_timezone"],
            },
        }

    # ── Create a single V2 alert ─────────────────────────────
    def create_alert(self, name: str, query_text: str,
                     source_col: str = "window_start") -> dict:
        body = self._alert_body(name, query_text, source_col)
        return self._api("POST", self.API, body)

    # ── Create all alerts from alerts.sql ────────────────────
    def create_alerts(self) -> list[dict]:
        alert_defs = self._parse_alerts_sql()
        created = []
        for defn in alert_defs:
            a = self.create_alert(f"Drift: {defn['name']}", defn["query_text"])
            print(f"  ✓ {defn['name']}  (ID: {a['id']})")
            created.append({"name": defn["name"], "id": a["id"]})
        return created

    # ── Create demo alert (always triggers) ───────────────────
    def create_demo_alert(self) -> dict:
        tbl = self.cfg["profile_table"]
        sql = (
            f"SELECT 'demo' AS alert_reason, current_timestamp() AS ts "
            f"FROM {tbl} WHERE column_name IS NOT NULL LIMIT 1"
        )
        a = self.create_alert(
            "Demo: Drift Alert Test (always triggers)", sql, source_col="ts",
        )
        print(f"  ✓ Demo alert  (ID: {a['id']})")
        return a

    # ── Delete alerts ────────────────────────────────────────
    def delete_alerts(self, alert_ids: list[str]) -> None:
        for aid in alert_ids:
            try:
                self._api("DELETE", f"{self.API}/{aid}")
                print(f"  ✗ Deleted {aid}")
            except Exception as e:
                print(f"  ! Failed {aid}: {e}")

    # ── List existing V2 alerts ──────────────────────────────
    def list_alerts(self) -> list[dict]:
        data = self._api("GET", self.API)
        return data.get("alerts", [])

## Run
Creates 3 drift-monitoring alerts + 1 demo alert.
The demo alert always triggers — manually **Refresh** it in the SQL UI during a presentation to fire the email.

In [0]:
mgr = AlertManager(ALERT_CONFIG)

# Clean up only alerts created by this notebook (match our naming prefix)
_OUR_PREFIXES = ("Drift:", "Demo: Drift Alert")
existing = mgr.list_alerts()
ours = [a for a in existing if a["display_name"].startswith(_OUR_PREFIXES)]
if ours:
    print(f"Removing {len(ours)} existing alert(s) from previous runs ...")
    mgr.delete_alerts([a["id"] for a in ours])
    skipped = len(existing) - len(ours)
    if skipped:
        print(f"  (kept {skipped} alert(s) owned by others)")

print("\nCreating drift alerts from alerts.sql ...")
created = mgr.create_alerts()

print("\nCreating demo alert ...")
demo = mgr.create_demo_alert()

print(f"\nDone — {len(created) + 1} V2 alerts created with {ALERT_CONFIG['notify_email']} subscribed.")

In [0]:
# Uncomment to tear down all V2 alerts created by this notebook
# mgr.delete_alerts([a["id"] for a in mgr.list_alerts()
#                     if a["display_name"].startswith(("Drift:", "Demo: Drift Alert"))])